# Experiment: Dual Readout Panel Prioritizer

Objective: prioritize a first-pass lab panel against the dual HbF plus alpha-globin/autophagy assay spec.

Success criteria:
- keep the required positive comparator, seed, mechanism comparator, support comparator, and hazard boundary covered;
- rank expansion items without promoting them as patient treatments;
- make the score formula explicit enough for a researcher or engineer to audit.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass

CORE_REQUIRED_TAGS = (
    "positive_hbf_control",
    "hbf_seed_for_hpfh_like_readouts",
    "autophagy_alpha_cleanup",
    "red_cell_metabolism_or_support",
    "hemolysis_hazard_boundary",
)

WEIGHTS = {
    "hbf": 0.30,
    "alpha_cleanup": 0.25,
    "autophagy": 0.15,
    "maturation_safety": 0.10,
    "hemolysis_gate": 0.10,
    "identity_access": 0.10,
    "patient_boundary_risk": -0.20,
}

## Plan

This is a computational design experiment, not wet-lab evidence.

Hypothesis: the five-item first lab panel remains coherent when re-scored against the newer dual-readout assay gate.

Metric: weighted dual-readout score plus mandatory coverage tags.

Decision rule:
- select one first-quote-ready item for each required coverage tag;
- keep hazard controls separate from therapy candidates;
- treat high-scoring but procurement- or identity-limited items as expansion lanes.


In [ ]:
@dataclass(frozen=True)
class Candidate:
    """One candidate or comparator for dual-readout panel design."""

    name: str
    role: str
    hbf: float
    alpha_cleanup: float
    autophagy: float
    maturation_safety: float
    hemolysis_gate: float
    identity_access: float
    patient_boundary_risk: float
    tags: frozenset[str]
    first_quote_ready: bool
    source_anchor: str
    boundary: str

    @property
    def dual_score(self) -> float:
        """Return the weighted dual-readout priority score."""
        score = 0.0
        score += self.hbf * WEIGHTS["hbf"]
        score += self.alpha_cleanup * WEIGHTS["alpha_cleanup"]
        score += self.autophagy * WEIGHTS["autophagy"]
        score += self.maturation_safety * WEIGHTS["maturation_safety"]
        score += self.hemolysis_gate * WEIGHTS["hemolysis_gate"]
        score += self.identity_access * WEIGHTS["identity_access"]
        score += self.patient_boundary_risk * WEIGHTS["patient_boundary_risk"]
        return score

In [ ]:
candidates = [
    Candidate(
        name="hydroxyurea",
        role="positive HbF comparator",
        hbf=5.0,
        alpha_cleanup=1.0,
        autophagy=1.0,
        maturation_safety=3.0,
        hemolysis_gate=4.0,
        identity_access=5.0,
        patient_boundary_risk=1.0,
        tags=frozenset({"positive_hbf_control", "hbf_readout_comparator"}),
        first_quote_ready=True,
        source_anchor="candidate scoring V0; hydroxyurea evidence map",
        boundary="clinician-supervised comparator only, not a new Nakafa Lab cure",
    ),
    Candidate(
        name="purified resveratrol",
        role="natural-product HbF seed",
        hbf=3.0,
        alpha_cleanup=1.5,
        autophagy=2.0,
        maturation_safety=3.0,
        hemolysis_gate=4.0,
        identity_access=4.0,
        patient_boundary_risk=1.5,
        tags=frozenset({"hbf_seed_for_hpfh_like_readouts"}),
        first_quote_ready=True,
        source_anchor="resveratrol HbF beta-thalassemia seed finding",
        boundary=(
            "screen seed only; must pass endogenous HbF, maturation, "
            "and hemolysis gates"
        ),
    ),
    Candidate(
        name="sirolimus",
        role="autophagy and alpha-globin cleanup comparator",
        hbf=3.5,
        alpha_cleanup=4.0,
        autophagy=5.0,
        maturation_safety=2.0,
        hemolysis_gate=3.0,
        identity_access=3.0,
        patient_boundary_risk=3.0,
        tags=frozenset({"autophagy_alpha_cleanup", "hbf_mechanism_comparator"}),
        first_quote_ready=True,
        source_anchor="PMID 37894732; sirolimus evidence map",
        boundary="immune-active monitor-only comparator; not self-treatment",
    ),
    Candidate(
        name="standardized 6-shogaol-rich ginger extract",
        role="red-cell support comparator",
        hbf=0.5,
        alpha_cleanup=1.0,
        autophagy=1.0,
        maturation_safety=3.5,
        hemolysis_gate=4.0,
        identity_access=3.5,
        patient_boundary_risk=1.5,
        tags=frozenset({"red_cell_metabolism_or_support"}),
        first_quote_ready=True,
        source_anchor="ginger shogaol red-cell support map",
        boundary=(
            "support comparator only unless HbF and alpha-globin endpoints improve"
        ),
    ),
    Candidate(
        name="melittin hazard control",
        role="hemolysis rejection threshold",
        hbf=0.0,
        alpha_cleanup=0.0,
        autophagy=0.0,
        maturation_safety=0.0,
        hemolysis_gate=5.0,
        identity_access=4.0,
        patient_boundary_risk=5.0,
        tags=frozenset({"hemolysis_hazard_boundary"}),
        first_quote_ready=True,
        source_anchor="melittin bee-venom hazard deep dive",
        boundary="hazard calibration only; blocked from therapy framing",
    ),
    Candidate(
        name="AMPKB1 / NRF2 pathway probe",
        role="target-discovery expansion",
        hbf=4.0,
        alpha_cleanup=2.5,
        autophagy=4.0,
        maturation_safety=2.0,
        hemolysis_gate=2.0,
        identity_access=1.0,
        patient_boundary_risk=3.0,
        tags=frozenset({"autophagy_alpha_cleanup", "hbf_seed_for_hpfh_like_readouts"}),
        first_quote_ready=False,
        source_anchor="AMPKB1 autophagy HbF bridge; PMID 41259521",
        boundary=(
            "target-discovery bridge; needs beta-thalassemia "
            "validation before promotion"
        ),
    ),
    Candidate(
        name="pyruvate kinase activator reference",
        role="red-cell metabolism expansion benchmark",
        hbf=1.0,
        alpha_cleanup=3.0,
        autophagy=1.0,
        maturation_safety=4.0,
        hemolysis_gate=4.0,
        identity_access=2.0,
        patient_boundary_risk=2.0,
        tags=frozenset({"red_cell_metabolism_or_support"}),
        first_quote_ready=False,
        source_anchor=(
            "pyruvate kinase red-cell metabolism benchmark; mitapivat clinical lane"
        ),
        boundary=(
            "benchmark or expansion reference only; access and legal "
            "procurement gate apply"
        ),
    ),
    Candidate(
        name="T-BDMC-like curcuminoid analog",
        role="identity-gated HbF chemistry expansion",
        hbf=3.5,
        alpha_cleanup=1.0,
        autophagy=1.5,
        maturation_safety=2.0,
        hemolysis_gate=2.0,
        identity_access=1.5,
        patient_boundary_risk=2.5,
        tags=frozenset({"hbf_seed_for_hpfh_like_readouts"}),
        first_quote_ready=False,
        source_anchor="T-BDMC identity resolution and cytotoxicity boundary",
        boundary=(
            "add only after exact identity, procurement, and hemolysis "
            "controls are solved"
        ),
    ),
]

In [ ]:
def best_ready_candidate(tag: str, pool: list[Candidate]) -> Candidate:
    """Return the highest-scoring first-quote-ready candidate for one tag."""
    eligible = [
        candidate
        for candidate in pool
        if candidate.first_quote_ready and tag in candidate.tags
    ]
    if not eligible:
        message = f"No first-quote-ready candidate covers required tag: {tag}"
        raise ValueError(message)
    return max(eligible, key=lambda candidate: candidate.dual_score)


def choose_core_panel(pool: list[Candidate]) -> list[Candidate]:
    """Choose one first-quote-ready item for every required coverage tag."""
    selected: list[Candidate] = []
    selected_names: set[str] = set()

    for tag in CORE_REQUIRED_TAGS:
        candidate = best_ready_candidate(tag, pool)
        if candidate.name in selected_names:
            continue
        selected.append(candidate)
        selected_names.add(candidate.name)

    return selected


def rank_expansion_items(
    pool: list[Candidate], core_panel: list[Candidate]
) -> list[Candidate]:
    """Return non-core candidates sorted by dual-readout score."""
    core_names = {candidate.name for candidate in core_panel}
    expansion = [candidate for candidate in pool if candidate.name not in core_names]
    return sorted(expansion, key=lambda candidate: candidate.dual_score, reverse=True)


def summarize_candidate(candidate: Candidate) -> dict[str, object]:
    """Return a compact table row for one candidate."""
    return {
        "candidate": candidate.name,
        "role": candidate.role,
        "dual_score": round(candidate.dual_score, 2),
        "first_quote_ready": candidate.first_quote_ready,
        "required_tags": sorted(candidate.tags & set(CORE_REQUIRED_TAGS)),
        "boundary": candidate.boundary,
    }

In [ ]:
core_panel = choose_core_panel(candidates)
expansion_items = rank_expansion_items(candidates, core_panel)

core_panel_summary = [summarize_candidate(candidate) for candidate in core_panel]
expansion_summary = [summarize_candidate(candidate) for candidate in expansion_items]

covered_tags = set().union(*(candidate.tags for candidate in core_panel))
missing_tags = sorted(set(CORE_REQUIRED_TAGS) - covered_tags)

result = {
    "decision": "first_quote_panel_stays_unchanged",
    "core_panel_size": len(core_panel),
    "missing_required_tags": missing_tags,
    "top_expansion_items": [item.name for item in expansion_items[:3]],
}

result

In [ ]:
core_panel_summary

In [ ]:
expansion_summary[:3]

## Results

The dual-readout prioritizer keeps the first quote panel unchanged:

1. hydroxyurea;
2. purified resveratrol;
3. sirolimus;
4. standardized 6-shogaol-rich ginger extract;
5. melittin hazard control.

The important change is not the core panel. The change is the explicit expansion logic: `AMPKB1` / `NRF2`, pyruvate-kinase activation, and `T-BDMC`-like curcuminoid chemistry can be added only when a qualified lab can support the required identity, procurement, alpha-globin, autophagy, maturation, and hemolysis gates.


## Interpretation

This notebook does not discover an active drug. It improves the next experiment design.

The core panel is still useful because it separates five roles that must not be blurred:

- a positive HbF comparator;
- a natural-product HbF seed;
- an autophagy and alpha-globin cleanup comparator;
- a red-cell support comparator;
- a hemolysis hazard boundary.

Promotion remains blocked until wet-lab data show HbF or F-cell improvement together with acceptable alpha-globin burden, autophagy context, erythroid maturation, viability, apoptosis, and mature red-cell hemolysis results.
